The following is the "beach" example from Skiena (Chapter 11.1)

In [1]:
import pandas as pd
df = pd.read_excel('beach_bayes.xlsx',index_col='Day',header=0)
df

,Outlook,Temperature,Humidity,Beach?
Day,,,,
1,Sunny,High,High,Yes
2,Sunny,High,Normal,Yes
3,Sunny,Low,Normal,No
4,Sunny,Mild,High,Yes
5,Rain,Mild,Normal,No
6,Rain,High,High,No
7,Rain,Low,Normal,No
8,Cloud,High,High,No
9,Cloud,High,Normal,Yes


We can compute the marginal probabilities.

In [2]:
n = len(df)
values = { feature : df[feature].unique() for feature in df.columns }

In [3]:
rows = list()
for feature in values:
    for value in values[feature]:
        row = { 
            'Feature' : feature, 
            'Value' : value,
            'Probability' : len(df.query(f'`{feature}` == @value'))/n
        }
        rows.append(row)
df_value_probability = pd.DataFrame(rows)
df_value_probability

,Feature,Value,Probability
0,Outlook,Sunny,0.4
1,Outlook,Rain,0.3
2,Outlook,Cloud,0.3
3,Temperature,High,0.5
4,Temperature,Low,0.2
5,Temperature,Mild,0.3
6,Humidity,High,0.4
7,Humidity,Normal,0.6
8,Beach?,Yes,0.4
9,Beach?,No,0.6


Note that the probabilities for a fixed class (e.g., for Outlook 0.4+0.3+0.3) sum to 1. This is as expected: if the values are listed exhaustively, we expect to see *some* value for each feature with absolute certainty.

We can then compute the marginal probabilities that P(feature has certain value | class) which we need to do in order to be able to construct a Naive Bayes classifier. In this case, the class label is the variable Beach? which we'll separate from the rest, and condition on that.

So, we compute two sets of probabilities: probability that we have a certain value assuming positive class label, and probability that we have the same value assuming negative class label.

In [4]:
classes = values['Beach?']
del values['Beach?']

In [5]:
rows = list()
n_beach = len(df.query('`Beach?` == "Yes"'))
n_nobeach = len(df.query('`Beach?` == "No"'))
for feature in values:
    for value in values[feature]:
        row = {
            'Feature' : feature,
            'Value' : value,
            'Beach' : len(df.query(f'{feature} == @value and `Beach?` == "Yes"')) / n_beach,
            'No Beach' : len(df.query(f'{feature} == @value and `Beach?` == "No"')) / n_nobeach,
        }
        rows.append(row)
df_beach_probability = pd.DataFrame(rows)
df_beach_probability

,Feature,Value,Beach,No Beach
0,Outlook,Sunny,0.75,0.166667
1,Outlook,Rain,0.00,0.500000
2,Outlook,Cloud,0.25,0.333333
3,Temperature,High,0.75,0.333333
4,Temperature,Low,0.00,0.333333
5,Temperature,Mild,0.25,0.333333
6,Humidity,High,0.50,0.333333
7,Humidity,Normal,0.50,0.666667


This time we see that if we sum the values for a feature, e.g., Outlook 0.75+0.25, they sum up to 1; again, we expect to see one value, regardless.

However, what is the overall probability of a beach day? We get this by summing all positive cases (and conversely for the negative). We had 10 cases so it should be the case that `n_beach + n_nobeach == n`.

In [6]:
n, n_beach, n_nobeach

(10, 4, 6)

As expected. So, the overall probability for a beach day os 4/10 and for no-beach day is 6/10.

To ease further analysis, let's convert these categorical features into numeric values (encode the values as numbers). We'll do this manually first.

In [7]:
def encode_feature(series):
    values = series.unique()
    value_to_ordinal = { v : i for (i,v) in enumerate(values) }
    ordinal_to_value = { i : v for (v,i) in value_to_ordinal.items() }
    return series.map(value_to_ordinal), ordinal_to_value

feature_encodings = dict()
df_encoded = dict()
for c in df.columns:
    encoded_series, ordinal_to_value = encode_feature(df[c])
    feature_encodings[c] = ordinal_to_value
    df_encoded[c] = encoded_series
df_encoded = pd.DataFrame(df_encoded)
df_encoded

,Outlook,Temperature,Humidity,Beach?
Day,,,,
1,0,0,0,0
2,0,0,1,0
3,0,1,1,1
4,0,2,0,0
5,1,2,1,1
6,1,0,0,1
7,1,1,1,1
8,2,0,0,1
9,2,0,1,0


We can also do the same using `sklearn.preprocessing` module. The class `OrdinalEncoder` can be used to automagically convert categorical values of *n* categories into (arbitrarily ordered) integers 0,1,..,*n*-1. The prediction target (in this case Beach?) should be encoded with `LabelEncoder`, although functionally there is little difference.

It is, however, customary to have the independent variables in a matrix like `X` and the dependent class labels or regression targets in a vector `y`.

In [8]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
oe = OrdinalEncoder()
le = LabelEncoder()
X = oe.fit_transform(df.iloc[:,:-1]) # we omit the class label
y = le.fit_transform(df['Beach?'])

/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_spar

In [9]:
X

array([[2., 0., 0.],
       [2., 0., 1.],
       [2., 1., 1.],
       [2., 2., 0.],
       [1., 2., 1.],
       [1., 0., 0.],
       [1., 1., 1.],
       [0., 0., 0.],
       [0., 0., 1.],
       [0., 2., 1.]])

In [10]:
y

array([1, 1, 0, 1, 0, 0, 0, 0, 1, 0])

The values have different meanings but this does not matter for categorical variables because the values should not be interpreted as having numerical values anyway (we only care if the numbers are the same or different when comparing two categories; categories have no order).

Category and class identities can be obtained from the variables `categories_` and `classes_`, and we can also use the `inverse_transform` function to recover the original class labels.

In [11]:
oe.categories_

[array(['Cloud', 'Rain', 'Sunny'], dtype=object),
 array(['High', 'Low', 'Mild'], dtype=object),
 array(['High', 'Normal'], dtype=object)]

In [12]:
le.classes_

array(['No', 'Yes'], dtype=object)

In [13]:
oe.inverse_transform(X)

array([['Sunny', 'High', 'High'],
       ['Sunny', 'High', 'Normal'],
       ['Sunny', 'Low', 'Normal'],
       ['Sunny', 'Mild', 'High'],
       ['Rain', 'Mild', 'Normal'],
       ['Rain', 'High', 'High'],
       ['Rain', 'Low', 'Normal'],
       ['Cloud', 'High', 'High'],
       ['Cloud', 'High', 'Normal'],
       ['Cloud', 'Mild', 'Normal']], dtype=object)

In [14]:
le.inverse_transform(y)

array(['Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No'],
      dtype=object)

We're now assuming that the features are *independent*. This means that we assume that P(outlook = sunny and temperature = high) = P(outlook = sunny)P(temperature = high), for example. This is the Naive Bayesian assumption.

Let's start by computing the probabilities manually. For example, by Bayes theorem, we have that P(Beach,(Sunny,Mild,High)) = P(Beach|(Sunny,Mild,High)) P(Sunny,Mild,High) = P((Sunny,Mild,High)|Beach)P(Beach). We can compute the same values for No Beach. We are interested in choosing the value that yields the highest probability for the class, so we want to compute P(Beach|(Sunny,Mild,High)) and P(No Beach|(Sunny,Mild,High)) and choose the appropriate class.

We can ignore the term P((Sunny,Mild,High)) because it's only a normalizing factor, independent of the  class.

By the Naive Bayes assumption, we can compute P((Sunny,Mild,High)|Beach) = P(Sunny|Beach)P(Mild|Beacg)P(High|Beach)P(Beach).

In [15]:
p = n_beach/n
for (feature,value) in zip(['Outlook','Temperature','Humidity'],['Sunny','Mild','High']):
    p *= df_beach_probability.query('Feature == @feature and Value == @value')['Beach'].iloc[0]
p

0.037500000000000006

We can do the same for no beach.

In [16]:
p = n_nobeach/n
for (feature,value) in zip(['Outlook','Temperature','Humidity'],['Sunny','Mild','High']):
    p *= df_beach_probability.query('Feature == @feature and Value == @value')['No Beach'].iloc[0]
p

0.011111111111111108

So this suggests this is a beach day. We can repeat this for all possible combinations.

In [17]:
from itertools import product
rows = list()
for feature_values in product(values['Outlook'],values['Temperature'],values['Humidity']):
    p_beach = n_beach/n
    p_nobeach = n_nobeach/n
    row = dict()
    for (feature,value) in zip(['Outlook','Temperature','Humidity'],feature_values):
        p_row = df_beach_probability.query('Feature == @feature and Value == @value')
        p_beach *= p_row['Beach'].iloc[0]
        p_nobeach *= p_row['No Beach'].iloc[0]
        row[feature] = value
    row['P beach'] = p_beach
    row['P no beach'] = p_nobeach
    row['Beach?'] = 'Yes' if p_beach >= p_nobeach else 'No'
    rows.append(row)
manual_result = pd.DataFrame(rows)
manual_result

,Outlook,Temperature,Humidity,P beach,P no beach,Beach?
0,Sunny,High,High,0.1125,0.011111,Yes
1,Sunny,High,Normal,0.1125,0.022222,Yes
2,Sunny,Low,High,0.0000,0.011111,No
3,Sunny,Low,Normal,0.0000,0.022222,No
4,Sunny,Mild,High,0.0375,0.011111,Yes
5,Sunny,Mild,Normal,0.0375,0.022222,Yes
6,Rain,High,High,0.0000,0.033333,No
7,Rain,High,Normal,0.0000,0.066667,No
8,Rain,Low,High,0.0000,0.033333,No
9,Rain,Low,Normal,0.0000,0.066667,No


Scikit Learn offers `CategoricalNB` that does the same. It has a so called `alpha` parameter for "smoothing" that, among other things, prevents problems that arise with zero observations (e.g., we never observe "yes" to a beach on a rainy day, so Outlook=Rainy dictates the answer).

In [18]:
from sklearn.naive_bayes import CategoricalNB
nb = CategoricalNB(alpha=0,force_alpha=True)
nb.fit(X,y)

/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/naive_bayes.py:1521: RuntimeWarning: divide by zero encountered in log
  np.log(smoothed_cat_count) - np.log(smoothed_class_count.reshape(-1, 1))


CategoricalNB(alpha=0, force_alpha=True)

The divide by zero warning above is related to the fact that in some cases the probabilities are zero. Let's repeat doing this for every case. We can transform the feature values into numerical values with the ordinal encoder that we had. We must be careful, however, to reshape the inputs appropriately because scikit learn expects observations to be the rows and features the columns, so we need the shape of input to be (18,3).

In [19]:
import numpy as np
X2 = [feature_values for feature_values in product(values['Outlook'],values['Temperature'],values['Humidity'])]
X2 = oe.transform(X2)

/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


This error arises from not using feature names which were implicitly encoded as the column names for the dataframe. The importance is in that using feature names makes them more robust against errors (e.g., if we misorder some value). We can bypass this by creating a dataframe (which might not be a bad idea in general).

In [20]:
X2 = pd.DataFrame([ 
    dict(zip(['Outlook','Temperature','Humidity'],feature_values)) for feature_values in product(values['Outlook'],values['Temperature'],values['Humidity'])
])
X2 = oe.transform(X2)
X2

/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/karppa/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_spar

array([[2., 0., 0.],
       [2., 0., 1.],
       [2., 1., 0.],
       [2., 1., 1.],
       [2., 2., 0.],
       [2., 2., 1.],
       [1., 0., 0.],
       [1., 0., 1.],
       [1., 1., 0.],
       [1., 1., 1.],
       [1., 2., 0.],
       [1., 2., 1.],
       [0., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 1.],
       [0., 2., 0.],
       [0., 2., 1.]])

We can predict the class labels by using the `predict` method. We can then recover the original labels by inverse transform.

In [21]:
y2 = nb.predict(X2)
le.inverse_transform(y2)

array(['Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'No',
       'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No'], dtype=object)

Let's construct a similar DataFrame.

In [22]:
rows = list()
for (feature_values,label) in zip(oe.inverse_transform(X2),le.inverse_transform(y2)):
    rows.append({
        'Outlook' : feature_values[0],
        'Temperature' : feature_values[1],
        'Humidity' : feature_values[2],
        'Beach?' : label
    })
nb_result = pd.DataFrame(rows)
nb_result

,Outlook,Temperature,Humidity,Beach?
0,Sunny,High,High,Yes
1,Sunny,High,Normal,Yes
2,Sunny,Low,High,No
3,Sunny,Low,Normal,No
4,Sunny,Mild,High,Yes
5,Sunny,Mild,Normal,Yes
6,Rain,High,High,No
7,Rain,High,Normal,No
8,Rain,Low,High,No
9,Rain,Low,Normal,No


We can see that the results match!

In [23]:
all(manual_result['Beach?'] == nb_result['Beach?'])

True

How do we deal with the zeros? One way is the Laplace or Lidstone smoothing where we add a value `alpha` to the probabilities (and adjust the divisors accordingly). The simplest case is when `alpha==1` which corresponds to the "add one discounting", as discussed by Skiena.

In [24]:
from sklearn.naive_bayes import CategoricalNB
nb = CategoricalNB(alpha=1)
nb.fit(X,y)

CategoricalNB(alpha=1)

Note that we no longer have a division by zero problem.

In [25]:
all(~((nb.predict(X2) == 1) ^ (nb_result['Beach?'] == 'Yes')))

True

There was no effect in this case. What if we try a larger value for alpha?

In [26]:
from sklearn.naive_bayes import CategoricalNB
nb = CategoricalNB(alpha=2)
nb.fit(X,y)
y3 = nb.predict(X2)

In [27]:
y2 == y3

array([ True,  True, False,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

In [28]:
oe.inverse_transform(X)[2,:]

array(['Sunny', 'Low', 'Normal'], dtype=object)

Now there is an difference. Let's try achieving the same manually.

In [34]:
rows = list()
n_beach = len(df.query('`Beach?` == "Yes"'))
n_nobeach = len(df.query('`Beach?` == "No"'))
alpha = 2
for feature in values:
    d = len(values[feature])
    for value in values[feature]:
        row = {
            'Feature' : feature,
            'Value' : value,
            'Beach' : (len(df.query(f'{feature} == @value and `Beach?` == "Yes"'))+alpha) / (n_beach+d*alpha),
            'No Beach' : (len(df.query(f'{feature} == @value and `Beach?` == "No"'))+alpha) / (n_nobeach+d*alpha),
        }
        rows.append(row)
df_beach_probability_smoothed = pd.DataFrame(rows)
df_beach_probability_smoothed

,Feature,Value,Beach,No Beach
0,Outlook,Sunny,0.5,0.250000
1,Outlook,Rain,0.2,0.416667
2,Outlook,Cloud,0.3,0.333333
3,Temperature,High,0.5,0.333333
4,Temperature,Low,0.2,0.333333
5,Temperature,Mild,0.3,0.333333
6,Humidity,High,0.5,0.400000
7,Humidity,Normal,0.5,0.600000


Now there's a non-zero chance for a beach day even on a rainy day.

In [35]:
rows = list()
for feature_values in product(values['Outlook'],values['Temperature'],values['Humidity']):
    p_beach = (n_beach+alpha)/(n+d*alpha)
    p_nobeach = (n_nobeach+alpha)/(n+d*alpha)
    row = dict()
    for (feature,value) in zip(['Outlook','Temperature','Humidity'],feature_values):
        p_row = df_beach_probability_smoothed.query('Feature == @feature and Value == @value')
        p_beach *= p_row['Beach'].iloc[0]
        p_nobeach *= p_row['No Beach'].iloc[0]
        row[feature] = value
    row['P beach'] = p_beach
    row['P no beach'] = p_nobeach
    row['Beach?'] = 'Yes' if p_beach >= p_nobeach else 'No'
    rows.append(row)
smoothed_manual_result = pd.DataFrame(rows)
smoothed_manual_result

,Outlook,Temperature,Humidity,P beach,P no beach,Beach?
0,Sunny,High,High,0.053571,0.019048,Yes
1,Sunny,High,Normal,0.053571,0.028571,Yes
2,Sunny,Low,High,0.021429,0.019048,Yes
3,Sunny,Low,Normal,0.021429,0.028571,No
4,Sunny,Mild,High,0.032143,0.019048,Yes
5,Sunny,Mild,Normal,0.032143,0.028571,Yes
6,Rain,High,High,0.021429,0.031746,No
7,Rain,High,Normal,0.021429,0.047619,No
8,Rain,Low,High,0.008571,0.031746,No
9,Rain,Low,Normal,0.008571,0.047619,No


In [36]:
all(smoothed_manual_result['Beach?'] == le.inverse_transform(y3))

True

This yields the same result!